In [ ]:

dataset_path = "data/raw/BraTS2024_small_dataset"

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = '  ' * level
    #print(f"{indent}{os.path.basename(root)}/")
    continue
    if level < 3:  # only show files up to 3 levels deep
        for f in files:
            #print(f"  {indent}{f}")
            continue

In [ ]:
import nibabel as nib
import numpy as np

# Pick the first case
case_path = "/kaggle/input/datasets/nguyenthanhkhanh/brats2024-small-dataset/BraTS2024_small_dataset/BraTS-GLI-02105-105"
case_id   = "BraTS-GLI-02105-105"

# Load T1c and segmentation
t1c = nib.load(f"{case_path}/{case_id}-t1c.nii")
seg = nib.load(f"{case_path}/{case_id}-seg.nii")

t1c_data = t1c.get_fdata()
seg_data = seg.get_fdata()

print("T1c shape      :", t1c_data.shape)
print("T1c voxel size :", t1c.header.get_zooms())
print("Seg shape      :", seg_data.shape)
print("Seg unique labels:", np.unique(seg_data))
print("Voxels per label:")
for label in np.unique(seg_data):
    print(f"  Label {int(label)}: {np.sum(seg_data == label)} voxels")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Pick the middle slice along each axis
mid_ax  = t1c_data.shape[0] // 2  # sagittal
mid_cor = t1c_data.shape[1] // 2  # coronal
mid_sag = t1c_data.shape[2] // 2  # axial

# Color map for segmentation overlay
# Label 2=edema(yellow), 3=enhancing(red), 4=necrotic(blue)
def make_overlay(seg_slice):
    h, w = seg_slice.shape
    overlay = np.zeros((h, w, 4), dtype=np.float32)  # RGBA
    overlay[seg_slice == 2] = [1.0, 0.9, 0.0, 0.6]  # yellow - edema
    overlay[seg_slice == 3] = [1.0, 0.2, 0.1, 0.8]  # red    - enhancing
    overlay[seg_slice == 4] = [0.1, 0.4, 1.0, 0.8]  # blue   - necrotic
    return overlay

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#0f0f0f')

views = [
    (t1c_data[mid_ax,  :, :], seg_data[mid_ax,  :, :], "Sagittal"),
    (t1c_data[:, mid_cor, :], seg_data[:, mid_cor, :], "Coronal"),
    (t1c_data[:, :, mid_sag], seg_data[:, :, mid_sag], "Axial"),
]

for ax, (t1c_slice, seg_slice, title) in zip(axes, views):
    # Normalize T1c for display
    t1c_norm = (t1c_slice - t1c_slice.min()) / (t1c_slice.max() - t1c_slice.min() + 1e-8)
    ax.imshow(t1c_norm.T, cmap='gray', origin='lower')
    ax.imshow(make_overlay(seg_slice).transpose(1, 0, 2), origin='lower')
    ax.set_title(title, color='white', fontsize=13)
    ax.axis('off')

# Legend
patches = [
    mpatches.Patch(color=[1.0, 0.9, 0.0], label='Edema (label 2)'),
    mpatches.Patch(color=[1.0, 0.2, 0.1], label='Enhancing tumor (label 3)'),
    mpatches.Patch(color=[0.1, 0.4, 1.0], label='Necrotic core (label 4)'),
]
fig.legend(handles=patches, loc='lower center', ncol=3,
           facecolor='#1a1a1a', labelcolor='white', fontsize=11, 
           framealpha=0.8, bbox_to_anchor=(0.5, -0.05))

plt.suptitle("BraTS-GLI-02105-105 — T1c with tumor overlay", 
             color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('slice_check.png', dpi=150, bbox_inches='tight', 
            facecolor='#0f0f0f')
plt.show()
print("Done — tumor visible in slices?")

In [ ]:
import plotly.graph_objects as go
from skimage.measure import marching_cubes

# ── 1. Extract brain surface from T1c (for context) ──────────────────────────
# Threshold: anything above 10% of max = brain tissue
brain_mask = t1c_data > (t1c_data.max() * 0.15)

# Smooth the brain mask slightly to reduce noise on surface
from scipy.ndimage import binary_closing, gaussian_filter
brain_mask = binary_closing(brain_mask, iterations=3)

verts_brain, faces_brain, _, _ = marching_cubes(brain_mask, level=0.5, step_size=3)

# ── 2. Extract tumor region surfaces ─────────────────────────────────────────
def extract_surface(seg_volume, label, step=1):
    mask = (seg_volume == label).astype(np.float32)
    if mask.sum() < 50:   # too small, skip
        return None
    verts, faces, _, _ = marching_cubes(mask, level=0.5, step_size=step)
    return verts, faces

edema     = extract_surface(seg_data, label=2)
enhancing = extract_surface(seg_data, label=3)
necrotic  = extract_surface(seg_data, label=4)

# ── 3. Helper: convert marching cubes output → Plotly Mesh3d ─────────────────
def make_mesh(verts, faces, color, opacity, name):
    x, y, z = verts[:, 0], verts[:, 1], verts[:, 2]
    i, j, k = faces[:, 0], faces[:, 1], faces[:, 2]
    return go.Mesh3d(
        x=x, y=y, z=z,
        i=i, j=j, k=k,
        color=color,
        opacity=opacity,
        name=name,
        showlegend=True,
        flatshading=False,
        lighting=dict(ambient=0.4, diffuse=0.8, specular=0.3, roughness=0.5),
        lightposition=dict(x=100, y=200, z=300)
    )

# ── 4. Build figure ───────────────────────────────────────────────────────────
traces = []

# Semi-transparent brain surface
traces.append(make_mesh(verts_brain, faces_brain,
                         color='#d4a57a', opacity=0.08, name='Brain'))

# Tumor regions
if edema:
    traces.append(make_mesh(*edema,
                             color='#FFD700', opacity=0.5, name='Edema (label 2)'))
if enhancing:
    traces.append(make_mesh(*enhancing,
                             color='#FF3322', opacity=0.9, name='Enhancing tumor (label 3)'))
if necrotic:
    traces.append(make_mesh(*necrotic,
                             color='#4488FF', opacity=0.85, name='Necrotic core (label 4)'))

fig = go.Figure(data=traces)

# ── 5. Layout ─────────────────────────────────────────────────────────────────
# Compute volume stats for annotation
voxel_vol_cm3 = 1.0  # 1mm³ per voxel → /1000 for cm³
edema_vol     = round(np.sum(seg_data == 2) / 1000, 2)
enhancing_vol = round(np.sum(seg_data == 3) / 1000, 2)
necrotic_vol  = round(np.sum(seg_data == 4) / 1000, 2)
total_vol     = round(edema_vol + enhancing_vol + necrotic_vol, 2)

fig.update_layout(
    title=dict(
        text=(f"CranioVision — BraTS-GLI-02105-105<br>"
              f"<sup>Edema: {edema_vol} cm³ | "
              f"Enhancing: {enhancing_vol} cm³ | "
              f"Necrotic: {necrotic_vol} cm³ | "
              f"Total tumor: {total_vol} cm³</sup>"),
        font=dict(size=16, color='white'),
        x=0.5
    ),
    paper_bgcolor='#0d0d0d',
    scene=dict(
        bgcolor='#0d0d0d',
        xaxis=dict(showticklabels=False, showgrid=False,
                   zeroline=False, title='', backgroundcolor='#0d0d0d'),
        yaxis=dict(showticklabels=False, showgrid=False,
                   zeroline=False, title='', backgroundcolor='#0d0d0d'),
        zaxis=dict(showticklabels=False, showgrid=False,
                   zeroline=False, title='', backgroundcolor='#0d0d0d'),
        camera=dict(eye=dict(x=1.8, y=1.8, z=1.0)),
        aspectmode='data'
    ),
    legend=dict(
        font=dict(color='white', size=12),
        bgcolor='rgba(30,30,30,0.8)',
        bordercolor='rgba(255,255,255,0.2)',
        borderwidth=1,
        x=0.01, y=0.99
    ),
    margin=dict(l=0, r=0, t=80, b=0),
    height=700,
)

fig.show()
print(f"\nTotal tumor volume : {total_vol} cm³")
print(f"  Edema            : {edema_vol} cm³")
print(f"  Enhancing tumor  : {enhancing_vol} cm³")
print(f"  Necrotic core    : {necrotic_vol} cm³")

In [ ]:
fig.write_html("/kaggle/working/cranovision_3d.html")
print("Done! Find the file in: /kaggle/working/cranovision_3d.html")